# Lab 05 · Generative Enrichment (Notebook Walkthrough)

**Concept.** The `genai_enrichment` profile adds a `ChatCompletionSkill` during indexing, so each document gets **summaries** and **keyword hints** generated at ingestion time. These become first-class retrieval cues that help both hybrid search and agentic planning.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': '<home>\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Ingest with generative enrichment

In [2]:
job = lab.ingest(skill_profile='genai_enrichment', reuse=True)
lab.chunk_overview(job)

Reusing existing 'genai_enrichment' ingestion (doc_id=0262a928, 412 chunks). Pass reuse=False to force a fresh run.


{'doc_id': '0262a928d6da473295bee04acb14c024',
 'skill_profile': 'genai_enrichment',
 'chunk_count': 412,
 'avg_tokens': 200.4,
 'max_tokens': 777,
 'chunks_with_summary': 412,
 'chunks_with_keyword_hints': 412,
 'chunks_with_image_description': 0}

Compared with Labs 03–04, `chunks_with_summary` and `chunks_with_keyword_hints` are now **non-zero** — the generative skill populated them.

## Step 3 — Inspect the generated enrichment

Show the summary + keyword hints that the `ChatCompletionSkill` produced.

In [3]:
import pandas as pd

chunks = lab.load_chunks(job)
enriched = [c for c in chunks if c.get('summary_text')]
pd.DataFrame([
    {
        'section': ' > '.join(c.get('section_path') or []) or '(root)',
        'summary': (c.get('summary_text') or '')[:160],
        'keyword_hints': ', '.join((c.get('keyword_hints') or [])[:6]),
    }
    for c in enriched[:6]
])

,section,summary,keyword_hints
0,Deep Excavation Design and Construction,"GEO Publication No. 1/2023, **Deep Excavation ...","GEO publication, Hong Kong, deep excavation, d..."
1,Deep Excavation Design and Construction,"GEO Publication No. 1/2023, **Deep Excavation ...","GEO publication, Hong Kong, deep excavation, d..."
2,Top Left,"GEO Publication No. 1/2023, **Deep Excavation ...","GEO publication, Hong Kong, deep excavation, d..."
3,Top Right,"GEO Publication No. 1/2023, **Deep Excavation ...","GEO publication, Hong Kong, deep excavation, d..."
4,Bottom Left,"GEO Publication No. 1/2023, **Deep Excavation ...","GEO publication, Hong Kong, deep excavation, d..."
5,Bottom Right,"GEO Publication No. 1/2023, **Deep Excavation ...","GEO publication, Hong Kong, deep excavation, d..."


## Step 4 — Query with Hybrid and Agentic

Generative cues help most on conceptual / multi-hop questions (Q2).

In [4]:
Q2 = 'Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?'

print('--- HYBRID ---')
resp_hybrid = lab.ask(Q2, job=job, retrieval_mode='hybrid', record_as='lab05_genai_hybrid')
lab.show_answer(resp_hybrid, max_citations=4)

--- HYBRID ---


[retrieval_mode=hybrid | scoring_profile=default | citations=8]

GEO Publication No. 1/2023, **Deep Excavation Design and Construction**, is a Hong Kong Geotechnical Engineering Office publication first published in December 2023. Loss of overall stability is likely to occur in excavations near a steeply-sloping site with a high groundwater table, or where a weak subsoil layer (e.g. loose sand or soft clay) is. [6]

Supporting evidence:
- In order to minimise disturbance to the adjacent ground due to the boring operation, the pressure of the compressed air should be carefully assessed and monitored, especially for sites with a high groundwater table and thick layers of loose fill, bouldery colluvium or rockfill. Trial boring is usually conducted and should be aimed to determine site-specific  [3]
- As discussed in Section 2.1.2, the differential piezometric pressures between marine clay and alluvial sand layers could lead to porewater pressure dissipation and associated consolidation s

In [5]:
print('--- AGENTIC ---')
resp_agentic = lab.ask(Q2, job=job, retrieval_mode='agentic', record_as='lab05_genai_agentic')
lab.show_answer(resp_agentic, max_citations=6)

--- AGENTIC ---


[retrieval_mode=agentic | scoring_profile=default | citations=8]

- **Key excavation risks** in high groundwater and clay layers include: - **Uplifting / base heave** when low-permeability soil overlies sand and artesian pressure exceeds overburden pressure; this can cause excessive upward movement and failure of the excavation support system.[1][2] - **Piping / hydraulic failure** from upward seepage into the excavation base, which can wash out soil, reduce passive support, and lead to sinkholes.[3][4] - **Overall instability** near steep slopes or where weak layers such as loose sand or soft clay exist below the wall toe.[5] - **Ground settlement** from groundwater drawdown, especially in fine-grained soils like marine clay, where dissipation of porewater pressure is slow and settlement may be delayed.[6][7] - **Excessive wall deformation / lateral movement** due to differences in soil and groundwater pressure between the excavated and retained sides.[8] - **Construction-related grou

## Takeaways

- Generative enrichment adds **summaries and keyword hints** at index time.
- These cues improve hybrid relevance and give the agentic planner better material to subquery against.

Next: **Lab 06** adds OCR + image analysis + language detection for diagram-heavy evidence.